In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from pathlib import Path

# Définition des chemins
OUT = Path('resultats')
chemin_dataset = OUT / 'dataset_prepare.csv'

# Chargement des données en forçant le format date
df = pd.read_csv(chemin_dataset, parse_dates=['acq_date'])

# Ajout d'une colonne 'annee' pour faciliter la séparation temporelle
df['annee'] = df['acq_date'].dt.year

print(f"Données chargées avec succès : {df.shape[0]} observations.")

Données chargées avec succès : 22788 observations.


In [2]:
# Définition de l'année charnière pour séparer les données
annee_limite = 2022

# Création des sous-ensembles (Train = Passé, Test = Futur)
train_df = df[df['annee'] < annee_limite].copy()
test_df = df[df['annee'] >= annee_limite].copy()

print(f"Jeu d'entraînement (Train) : {train_df.shape[0]} observations (Années < {annee_limite})")
print(f"Jeu de test (Test) : {test_df.shape[0]} observations (Années >= {annee_limite})")

Jeu d'entraînement (Train) : 11783 observations (Années < 2022)
Jeu de test (Test) : 11005 observations (Années >= 2022)


In [3]:
# Définition de l'année charnière pour séparer les données
annee_limite = 2022

# Création des sous-ensembles (Train = Passé, Test = Futur)
train_df = df[df['annee'] < annee_limite].copy()
test_df = df[df['annee'] >= annee_limite].copy()

print(f"Jeu d'entraînement (Train) : {train_df.shape[0]} observations (Années < {annee_limite})")
print(f"Jeu de test (Test) : {test_df.shape[0]} observations (Années >= {annee_limite})")

Jeu d'entraînement (Train) : 11783 observations (Années < 2022)
Jeu de test (Test) : 11005 observations (Années >= 2022)


In [4]:
# Le vrai nom de votre colonne cible est 'feu'
colonne_cible = 'feu'

# On liste toutes les colonnes qui ne sont PAS des variables prédictives
# On enlève les dates, les identifiants, et les métadonnées de la grille
colonnes_a_ignorer = [
    'acq_date', 'date', 'annee',
    'id_observation', 'lat_grille', 'lon_grille',
    'origine_meteo', 'statut_geo', 'distance_grille_km',
    colonne_cible
]

# On sélectionne automatiquement les colonnes restantes comme features
features = [col for col in df.columns if col not in colonnes_a_ignorer]

# Séparation X (Variables) et y (Cible) pour l'entraînement
X_train = train_df[features]
y_train = train_df[colonne_cible]

# Séparation X (Variables) et y (Cible) pour le test
X_test = test_df[features]
y_test = test_df[colonne_cible]

print(f"Nombre de features utilisées pour la prédiction : {len(features)}")
print(f"Liste des features : {features}")

Nombre de features utilisées pour la prédiction : 8
Liste des features : ['lat_c', 'lon_c', 'mois', 'jour_annee', 'temperature_2m_max', 'precipitation_sum', 'wind_speed_10m_max', 'humidite_moy']


In [5]:
# Initialisation du scaler
scaler = StandardScaler()

# Apprentissage de l'échelle sur le Train ET transformation
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=features, index=X_train.index)

# Application de la transformation sur le Test (sans réapprendre l'échelle)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=features, index=X_test.index)

print("Normalisation (StandardScaler) terminée avec succès !")

Normalisation (StandardScaler) terminée avec succès !


In [6]:
# Sauvegarde des matrices transformées et des cibles
X_train_scaled.to_csv(OUT / 'X_train_scaled.csv', index=False)
X_test_scaled.to_csv(OUT / 'X_test_scaled.csv', index=False)

y_train.to_csv(OUT / 'y_train.csv', index=False)
y_test.to_csv(OUT / 'y_test.csv', index=False)

print("Toutes les données préparées ont été sauvegardées dans le dossier 'resultats' !")

Toutes les données préparées ont été sauvegardées dans le dossier 'resultats' !


In [7]:


from xgboost import XGBClassifier
from sklearn.metrics import classification_report

print("Entraînement du modèle XGBoost en cours...")

# Calcul du ratio pour forcer le modèle à prêter attention aux feux
# ratio = (Nombre de 'Pas de Feu') / (Nombre de 'Feu') dans le train set
ratio_desequilibre = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    scale_pos_weight=ratio_desequilibre, # Pénalise lourdement les erreurs sur la classe 1 (Feu)
    random_state=42,
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6
)

# Apprentissage
xgb_model.fit(X_train, y_train)

# Prédiction
y_pred_xgb = xgb_model.predict(X_test)

print("\n--- RAPPORT DE CLASSIFICATION (XGBOOST) ---\n")
print(classification_report(y_test, y_pred_xgb, target_names=['Pas de Feu (0)', 'Feu (1)']))

Entraînement du modèle XGBoost en cours...

--- RAPPORT DE CLASSIFICATION (XGBOOST) ---

                precision    recall  f1-score   support

Pas de Feu (0)       0.69      0.58      0.63      7127
       Feu (1)       0.41      0.53      0.46      3878

      accuracy                           0.56     11005
     macro avg       0.55      0.55      0.55     11005
  weighted avg       0.59      0.56      0.57     11005

